# Cross-Cell-Line UMAP Visualization

Investigate what the model predicts for unseen perturbations in the eval cell line (A549).

**Question**: When anchor (SW48) has seen drug X but eval (A549) has not,
is the model predicting SW48-like responses for A549?

**Plots**:
- For Set B drugs (unseen in A549): ctrl vs true vs pred (from A1 and B1)
- Also overlay SW48 true perturbed cells to see if B1's predictions resemble SW48

In [1]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import cloudpickle
import optax
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from functools import partial
from scipy.spatial.distance import cdist

from scaleflow.data import AnnDataLocation, DataManager
from scaleflow.data._dataloader import InMemorySampler, ValidationSampler, CombinedSampler
from scaleflow.model import ScaleFlow
from scaleflow.utils import match_linear

/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/_utils/__init__.py:35: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/icb/xiaotong.fu/miniconda3/envs/cstm_scvi_env/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


ModuleNotFoundError: No module named 'scaleflow'

## 1. Config

In [ ]:
DATA_DIR = '/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/unipert'
OUTPUT_BASE = '/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/outputs/crosscellline'

CELLLINE_FILES = {
    'A549':   'tahoe_a549_w_emb.h5ad',
    'SW48':   'tahoe_sw48_w_emb.h5ad',
    'PANC-1': 'tahoe_panc_1_w_emb.h5ad',
}

# Experiments to compare
EXPERIMENTS = {
    'A1': {'anchor': [],       'eval': ['A549'], 'ckpt': os.path.join(OUTPUT_BASE, 'A1', 'best_params.pkl')},
    'B1': {'anchor': ['SW48'], 'eval': ['A549'], 'ckpt': os.path.join(OUTPUT_BASE, 'B1', 'best_params.pkl')},
}

DRUG_SPLIT_SEED = 42
MAX_CELLS = 200000
SEED = 42
N_FOCUS_DRUGS = 5  # Number of Set B drugs to visualize
UMAP_SUBSAMPLE = 500  # cells per group in UMAP

for exp_id, cfg in EXPERIMENTS.items():
    print(f'{exp_id}: ckpt exists = {os.path.exists(cfg["ckpt"])}')

## 2. Load data and compute drug split

In [ ]:
def mark_control(adata):
    adata.obs['control'] = (adata.obs['drug_0'] == 'control')
    if 'drug_1' in adata.obs.columns:
        adata.obs['control'] = adata.obs['control'] & (adata.obs['drug_1'] == 'control')
    return adata

def resample_cells(adata, target_cells, rng):
    if target_cells >= adata.n_obs:
        return adata
    fraction = target_cells / adata.n_obs
    indices = []
    for _, group in adata.obs.groupby(['drug_0'], observed=True):
        group_indices = group.index.tolist()
        n_sample = max(1, int(len(group_indices) * fraction))
        sampled = rng.choice(group_indices, size=n_sample, replace=False)
        indices.extend(sampled)
    return adata[indices].copy()

def filter_drugs(adata, keep_drugs, keep_control=True):
    mask = adata.obs['drug_0'].isin(keep_drugs)
    if keep_control:
        mask = mask | adata.obs['control']
    return adata[mask].copy()

# Get global drug list
rng = np.random.default_rng(SEED)
all_drugs = set()
for cl_name, fname in CELLLINE_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    adata_backed = ad.read_h5ad(path, backed='r')
    drugs = [d for d in adata_backed.obs['drug_0'].unique() if d != 'control']
    all_drugs.update(drugs)
    print(f'{cl_name}: {len(drugs)} drugs, {adata_backed.n_obs:,} cells')
    del adata_backed

# Drug split (same as training)
drugs_sorted = sorted(all_drugs)
rng_split = np.random.default_rng(DRUG_SPLIT_SEED)
idx = rng_split.permutation(len(drugs_sorted))
half = len(drugs_sorted) // 2
set_a = {drugs_sorted[i] for i in idx[:half]}
set_b = {drugs_sorted[i] for i in idx[half:]}
print(f'\nDrug split: Set A = {len(set_a)}, Set B = {len(set_b)}')

In [ ]:
# Load A549 full data (for eval: Set A train, Set B test)
print('Loading A549...')
a549_full = sc.read_h5ad(os.path.join(DATA_DIR, CELLLINE_FILES['A549']))
a549_full = mark_control(a549_full)

a549_train = filter_drugs(a549_full, set_a, keep_control=True)
a549_train = resample_cells(a549_train, MAX_CELLS, rng)
a549_eval = filter_drugs(a549_full, set_b, keep_control=True)
print(f'  A549 train (Set A): {a549_train.n_obs:,} cells')
print(f'  A549 eval (Set B):  {a549_eval.n_obs:,} cells')

# Load SW48 full data (anchor: all drugs)
print('Loading SW48...')
sw48_full = sc.read_h5ad(os.path.join(DATA_DIR, CELLLINE_FILES['SW48']))
sw48_full = mark_control(sw48_full)
sw48_full = resample_cells(sw48_full, MAX_CELLS, rng)
print(f'  SW48 (all drugs):   {sw48_full.n_obs:,} cells')

del a549_full
gc.collect()

## 3. Pick focus drugs from Set B
Choose drugs that have enough cells in both A549 and SW48.

In [ ]:
# Find Set B drugs with good cell counts in both cell lines
a549_drug_counts = a549_eval.obs[~a549_eval.obs['control']].groupby('drug_0').size()
sw48_drug_counts = sw48_full.obs[~sw48_full.obs['control']].groupby('drug_0').size()

set_b_drugs = sorted(set_b)
drug_info = []
for d in set_b_drugs:
    n_a549 = a549_drug_counts.get(d, 0)
    n_sw48 = sw48_drug_counts.get(d, 0)
    if n_a549 >= 50 and n_sw48 >= 50:
        drug_info.append({'drug': d, 'n_a549': n_a549, 'n_sw48': n_sw48})

drug_df = pd.DataFrame(drug_info).sort_values('n_a549', ascending=False)
print(f'Set B drugs with >=50 cells in both A549 and SW48: {len(drug_df)}')
print(drug_df.head(10))

# Pick focus drugs
focus_drugs = drug_df['drug'].head(N_FOCUS_DRUGS).tolist()
print(f'\nFocus drugs: {focus_drugs}')

## 4. Build models and generate predictions

In [ ]:
def build_model_and_predict(exp_id, exp_cfg, a549_train, a549_eval, sw48_full, set_a, set_b, focus_drugs, seed=42):
    """Recreate model, load checkpoint, predict on focus drugs."""
    print(f'\n{"=" * 60}')
    print(f'Experiment: {exp_id}')
    print(f'  Anchor: {exp_cfg["anchor"]}, Eval: {exp_cfg["eval"]}')
    
    rng = np.random.default_rng(seed)
    
    # Build training data (same as training script)
    train_adatas = {}
    if exp_cfg['anchor']:
        for cl in exp_cfg['anchor']:
            if cl == 'SW48':
                train_adatas[cl] = sw48_full
    train_adatas['A549'] = a549_train
    
    # DataManager
    adl = AnnDataLocation()
    data_manager = DataManager(
        dist_flag_key='control',
        src_dist_keys=['cell_line'],
        tgt_dist_keys=['drug_0'],
        rep_keys={'cell_line': 'cell_line_ccle_embeddings', 'drug_0': 'drug_0_embeddings'},
        data_location=adl.obsm['X_state'],
    )
    
    # Prepare grouped distributions for training (needed to init model)
    train_gds = {}
    for cl, adata in train_adatas.items():
        train_gds[cl] = data_manager.prepare_data(adata)
    
    # Create sampler and get sample batch
    train_samplers = {}
    for cl in train_adatas:
        train_samplers[cl] = InMemorySampler(
            train_gds[cl], np.random.default_rng(seed + hash(cl) % 1000), batch_size=512,
        )
    if len(train_samplers) > 1:
        sampler = CombinedSampler(samplers=train_samplers, rng=np.random.default_rng(seed))
    else:
        sampler = list(train_samplers.values())[0]
    sampler.init_sampler()
    sample_batch = sampler.sample()
    
    # Create model (same architecture as training)
    sf = ScaleFlow(solver='otfm')
    lr_schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=1e-3, warmup_steps=2000, decay_steps=200000, end_value=1e-5,
    )
    match_fn = partial(match_linear, epsilon=10.0)
    sf.prepare_model(
        sample_batch=sample_batch,
        max_combination_length=1,
        conditioning='concatenation',
        decoder_dims=(2048, 2048, 2048),
        hidden_dims=(2048, 2048, 2048),
        conditioning_kwargs={},
        match_fn=match_fn,
        probability_path={'constant_noise': 0.1},
        optimizer=optax.MultiSteps(optax.adam(learning_rate=lr_schedule), 20),
    )
    
    # Load checkpoint
    ckpt_path = exp_cfg['ckpt']
    print(f'  Loading checkpoint: {ckpt_path}')
    with open(ckpt_path, 'rb') as f:
        best_params = cloudpickle.load(f)
    sf._solver.vf_state = sf._solver.vf_state.replace(params=best_params)
    sf._solver.vf_state_inference = sf._solver.vf_state_inference.replace(params=best_params)
    print('  Checkpoint loaded.')
    
    # Prepare eval data and predict on focus drugs
    gd_eval = data_manager.prepare_data(a549_eval)
    eval_sampler = ValidationSampler(gd_eval, n_conditions_on_log_iteration=None,
                                      n_conditions_on_train_end=None, seed=seed)
    if not eval_sampler.initialized:
        eval_sampler.init_sampler()
    
    annotation = eval_sampler._data.annotation
    tgt_labels = annotation.tgt_dist_idx_to_labels
    src_labels = annotation.src_dist_idx_to_labels
    tgt_to_src = eval_sampler._tgt_to_src
    gd_data = eval_sampler._data.data
    
    results = {}  # {drug: {'pred': array, 'true': array, 'ctrl': array}}
    for tgt_idx in gd_data.conditions:
        src_idx = tgt_to_src.get(tgt_idx)
        if src_idx is None:
            continue
        drug = tgt_labels[tgt_idx][0] if tgt_idx in tgt_labels else str(tgt_idx)
        if drug not in focus_drugs:
            continue
        
        print(f'  Predicting {drug}...')
        pred = np.array(sf._solver.predict(gd_data.src_data[src_idx], gd_data.conditions[tgt_idx]))
        tgt = np.array(gd_data.tgt_data[tgt_idx])
        src = np.array(gd_data.src_data[src_idx])
        results[drug] = {'pred': pred, 'true': tgt, 'ctrl': src}
    
    del sf, sampler, train_samplers, train_gds
    gc.collect()
    return results

In [ ]:
# Get SW48 true responses for focus drugs (for comparison)
adl = AnnDataLocation()
dm_sw48 = DataManager(
    dist_flag_key='control',
    src_dist_keys=['cell_line'],
    tgt_dist_keys=['drug_0'],
    rep_keys={'cell_line': 'cell_line_ccle_embeddings', 'drug_0': 'drug_0_embeddings'},
    data_location=adl.obsm['X_state'],
)
gd_sw48 = dm_sw48.prepare_data(sw48_full)
sw48_sampler = ValidationSampler(gd_sw48, n_conditions_on_log_iteration=None,
                                  n_conditions_on_train_end=None, seed=SEED)
if not sw48_sampler.initialized:
    sw48_sampler.init_sampler()

sw48_ann = sw48_sampler._data.annotation
sw48_tgt_labels = sw48_ann.tgt_dist_idx_to_labels
sw48_gd = sw48_sampler._data.data
sw48_tgt_to_src = sw48_sampler._tgt_to_src

sw48_responses = {}  # {drug: {'true': array, 'ctrl': array}}
for tgt_idx in sw48_gd.conditions:
    src_idx = sw48_tgt_to_src.get(tgt_idx)
    if src_idx is None:
        continue
    drug = sw48_tgt_labels[tgt_idx][0] if tgt_idx in sw48_tgt_labels else str(tgt_idx)
    if drug in focus_drugs:
        sw48_responses[drug] = {
            'true': np.array(sw48_gd.tgt_data[tgt_idx]),
            'ctrl': np.array(sw48_gd.src_data[src_idx]),
        }

print(f'SW48 responses loaded for: {list(sw48_responses.keys())}')

In [ ]:
# Generate predictions for each experiment
all_results = {}
for exp_id, exp_cfg in EXPERIMENTS.items():
    all_results[exp_id] = build_model_and_predict(
        exp_id, exp_cfg, a549_train, a549_eval, sw48_full,
        set_a, set_b, focus_drugs, seed=SEED,
    )

## 5. Spotlight UMAPs

For each focus drug, show:
- A549 ctrl (blue)
- A549 true perturbed (red)
- A549 predicted by A1 - no anchor (green)
- A549 predicted by B1 - SW48 anchor (orange)
- SW48 true perturbed (purple) — is B1 prediction close to this?

In [ ]:
PALETTE = {
    'A549 ctrl': '#4878CF',
    'A549 true': '#D65F5F',
    'A1 pred (no anchor)': '#6ACC65',
    'B1 pred (SW48 anchor)': '#FF8C00',
    'SW48 true': '#9467BD',
}

def subsample(arr, n, rng):
    if len(arr) <= n:
        return arr
    return arr[rng.choice(len(arr), n, replace=False)]


def plot_drug_umap(drug, all_results, sw48_responses, n_sub=UMAP_SUBSAMPLE, seed=42):
    """Build joint UMAP for one drug across experiments."""
    rng = np.random.default_rng(seed)
    
    parts, labels = [], []
    
    # A549 ctrl
    ctrl = all_results['A1'][drug]['ctrl']
    ctrl_sub = subsample(ctrl, n_sub, rng)
    parts.append(ctrl_sub)
    labels.extend(['A549 ctrl'] * len(ctrl_sub))
    
    # A549 true
    true = all_results['A1'][drug]['true']
    true_sub = subsample(true, n_sub, rng)
    parts.append(true_sub)
    labels.extend(['A549 true'] * len(true_sub))
    
    # A1 prediction (no anchor)
    pred_a1 = all_results['A1'][drug]['pred']
    pred_a1_sub = subsample(pred_a1, n_sub, rng)
    parts.append(pred_a1_sub)
    labels.extend(['A1 pred (no anchor)'] * len(pred_a1_sub))
    
    # B1 prediction (SW48 anchor)
    if drug in all_results.get('B1', {}):
        pred_b1 = all_results['B1'][drug]['pred']
        pred_b1_sub = subsample(pred_b1, n_sub, rng)
        parts.append(pred_b1_sub)
        labels.extend(['B1 pred (SW48 anchor)'] * len(pred_b1_sub))
    
    # SW48 true perturbed
    if drug in sw48_responses:
        sw48_true = sw48_responses[drug]['true']
        sw48_sub = subsample(sw48_true, n_sub, rng)
        parts.append(sw48_sub)
        labels.extend(['SW48 true'] * len(sw48_sub))
    
    X = np.vstack(parts).astype(np.float32)
    adata_u = ad.AnnData(X=X, obs=pd.DataFrame({'group': labels}))
    adata_u.obs_names = [str(i) for i in range(len(adata_u))]
    
    n_pcs = min(50, adata_u.n_obs - 1, adata_u.n_vars - 1)
    sc.pp.pca(adata_u, n_comps=n_pcs)
    sc.pp.neighbors(adata_u, n_pcs=n_pcs)
    sc.tl.umap(adata_u)
    
    coords = adata_u.obsm['X_umap']
    groups = adata_u.obs['group'].values
    
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    
    # Plot each group
    plot_order = ['A549 ctrl', 'SW48 true', 'A549 true', 'A1 pred (no anchor)', 'B1 pred (SW48 anchor)']
    for group_name in plot_order:
        mask = groups == group_name
        if mask.sum() == 0:
            continue
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=PALETTE[group_name], s=8, alpha=0.5, label=f'{group_name} [{mask.sum()}]',
                   rasterized=True)
        # Centroid star
        mu = coords[mask].mean(axis=0)
        ax.scatter(mu[0], mu[1], c=PALETTE[group_name], marker='*',
                   s=300, edgecolors='k', linewidths=0.5, zorder=10)
    
    ax.set_title(f'Drug: {drug} (Set B — unseen in A549)', fontsize=13)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    ax.legend(fontsize=9, loc='best', markerscale=2)
    ax.axis('equal')
    plt.tight_layout()
    return fig, adata_u

In [ ]:
# Plot each focus drug
out_dir = os.path.join(OUTPUT_BASE, 'umap_analysis')
os.makedirs(out_dir, exist_ok=True)

for drug in focus_drugs:
    if drug not in all_results['A1']:
        print(f'Skipping {drug} (not in A1 results)')
        continue
    print(f'\nPlotting {drug}...')
    fig, adata_u = plot_drug_umap(drug, all_results, sw48_responses)
    fig.savefig(os.path.join(out_dir, f'spotlight_{drug}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    del adata_u; gc.collect()

print(f'\nSaved to {out_dir}')

## 6. Quantitative: centroid distances

For each Set B drug, compute distances between prediction centroids and
SW48 true vs A549 true centroids. If B1 is transplanting SW48 responses,
B1 predictions should be closer to SW48 true than to A549 true.

In [ ]:
rows = []
for drug in all_results['A1']:
    if drug not in all_results.get('B1', {}) or drug not in sw48_responses:
        continue
    
    a549_true_mean = all_results['A1'][drug]['true'].mean(axis=0)
    a549_ctrl_mean = all_results['A1'][drug]['ctrl'].mean(axis=0)
    sw48_true_mean = sw48_responses[drug]['true'].mean(axis=0)
    
    pred_a1_mean = all_results['A1'][drug]['pred'].mean(axis=0)
    pred_b1_mean = all_results['B1'][drug]['pred'].mean(axis=0)
    
    rows.append({
        'drug': drug,
        # A1 distances
        'A1_to_a549_true': np.linalg.norm(pred_a1_mean - a549_true_mean),
        'A1_to_ctrl': np.linalg.norm(pred_a1_mean - a549_ctrl_mean),
        # B1 distances
        'B1_to_a549_true': np.linalg.norm(pred_b1_mean - a549_true_mean),
        'B1_to_sw48_true': np.linalg.norm(pred_b1_mean - sw48_true_mean),
        'B1_to_ctrl': np.linalg.norm(pred_b1_mean - a549_ctrl_mean),
        # Reference
        'a549_true_to_sw48_true': np.linalg.norm(a549_true_mean - sw48_true_mean),
        'a549_true_to_ctrl': np.linalg.norm(a549_true_mean - a549_ctrl_mean),
    })

dist_df = pd.DataFrame(rows)
print(f'{len(dist_df)} drugs analyzed')

# Key question: is B1 pred closer to SW48 true than A549 true?
dist_df['B1_closer_to_sw48'] = dist_df['B1_to_sw48_true'] < dist_df['B1_to_a549_true']
print(f'\nB1 pred closer to SW48 true than A549 true: {dist_df["B1_closer_to_sw48"].mean():.1%}')
print(f'\nMean centroid distances:')
print(f'  A1 pred → A549 true:  {dist_df["A1_to_a549_true"].mean():.3f}')
print(f'  B1 pred → A549 true:  {dist_df["B1_to_a549_true"].mean():.3f}')
print(f'  B1 pred → SW48 true:  {dist_df["B1_to_sw48_true"].mean():.3f}')
print(f'  A549 true → SW48 true: {dist_df["a549_true_to_sw48_true"].mean():.3f}')

dist_df.to_csv(os.path.join(out_dir, 'centroid_distances.csv'), index=False)
dist_df.head(10)

In [ ]:
# Scatter: B1 distance to SW48 true vs B1 distance to A549 true
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: B1 pred — closer to SW48 or A549?
ax = axes[0]
ax.scatter(dist_df['B1_to_a549_true'], dist_df['B1_to_sw48_true'], s=30, alpha=0.7)
lim = max(dist_df[['B1_to_a549_true', 'B1_to_sw48_true']].max())
ax.plot([0, lim], [0, lim], 'r--', lw=0.8, label='equal distance')
ax.set_xlabel('B1 pred → A549 true')
ax.set_ylabel('B1 pred → SW48 true')
ax.set_title('B1: closer to SW48 or A549?\n(below diagonal = closer to SW48)')
ax.legend()

# Right: A1 vs B1 distance to A549 true
ax = axes[1]
ax.scatter(dist_df['A1_to_a549_true'], dist_df['B1_to_a549_true'], s=30, alpha=0.7)
lim = max(dist_df[['A1_to_a549_true', 'B1_to_a549_true']].max())
ax.plot([0, lim], [0, lim], 'r--', lw=0.8, label='equal')
ax.set_xlabel('A1 pred → A549 true (no anchor)')
ax.set_ylabel('B1 pred → A549 true (SW48 anchor)')
ax.set_title('Is B1 worse than A1?\n(above diagonal = B1 worse)')
ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(out_dir, 'centroid_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()